# Clean Fight Data File

Outputs files
- all_fights.csv
- fighter_career_stats.csv

In [7]:
import pandas as pd
import numpy as np

# Input/output paths
input_path = "../data_raw/ufc_fights.csv"
output_path = "all_fights.csv"

raw_fights = pd.read_csv(input_path)

# Drop unnecessary metadata columns
cols_to_drop = [
    "event_name",
    "event_url",
    "fight_url",
    "location",
    "attendance",
    "fight_type",
    "R_status",
    "B_status",
    "referee",
    "details"
]

raw_fights = raw_fights.drop(columns=cols_to_drop, errors="ignore")

# Convert percentage columns from strings like "45%" to decimals like 0.45
# Empty values, NaN, and "---" become 0
pct_cols = [
    "R_SIG_STR_pct",
    "B_SIG_STR_pct",
    "R_TD_pct",
    "B_TD_pct"
]

for col in pct_cols:
    raw_fights[col] = (
        raw_fights[col]
        .astype(str)
        .str.strip()
        .str.replace("%", "", regex=False)
        .replace(["", "---", "nan", "None"], "0")
    )

    raw_fights[col] = pd.to_numeric(raw_fights[col], errors="coerce").fillna(0)

# Save updated cleaned file
raw_fights.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print("Shape:", raw_fights.shape)

Saved: all_fights.csv
Shape: (8705, 64)


In [8]:
# Split fight-level rows into fighter-level rows, including opponent landed/attempted stats

shared_cols = ["winner", "win_by", "last_round", "last_round_time", "format", "date"]

# Fighter-side stats from the new dataset
# These are already split into landed/attempted columns where needed
stat_cols = [
    "KD",
    "SIG_STR_landed",
    "SIG_STR_att",
    "SIG_STR_pct",
    "TOTAL_STR_landed",
    "TOTAL_STR_att",
    "TD_landed",
    "TD_att",
    "TD_pct",
    "SUB_ATT",
    "REV",
    "CTRL",
    "HEAD_landed",
    "HEAD_att",
    "BODY_landed",
    "BODY_att",
    "LEG_landed",
    "LEG_att",
    "DISTANCE_landed",
    "DISTANCE_att",
    "CLINCH_landed",
    "CLINCH_att",
    "GROUND_landed",
    "GROUND_att"
]

# Opponent stats needed later for absorbed/defensive metrics
# No percentages needed here because defense is calculated from opponent landed/attempted totals
opp_stat_cols = [
    "SIG_STR_landed",
    "SIG_STR_att",
    "TOTAL_STR_landed",
    "TOTAL_STR_att",
    "TD_landed",
    "TD_att",
    "HEAD_landed",
    "HEAD_att",
    "BODY_landed",
    "BODY_att",
    "LEG_landed",
    "LEG_att",
    "DISTANCE_landed",
    "DISTANCE_att",
    "CLINCH_landed",
    "CLINCH_att",
    "GROUND_landed",
    "GROUND_att"
]

def opp_name(col):
    """
    Example:
    SIG_STR_landed -> SIG_STR_opp_landed
    SIG_STR_att    -> SIG_STR_opp_att
    """
    if col.endswith("_landed"):
        return col.replace("_landed", "_opp_landed")
    elif col.endswith("_att"):
        return col.replace("_att", "_opp_att")
    else:
        return f"{col}_opp"


# Red fighter rows
r_rows = raw_fights[
    ["R_fighter", "B_fighter", "R_DOB"]
    + shared_cols
    + [f"R_{col}" for col in stat_cols]
    + [f"B_{col}" for col in opp_stat_cols]
].copy()

r_rows = r_rows.rename(columns={
    "R_fighter": "fighter_name",
    "B_fighter": "opponent",
    "R_DOB": "DOB",
    **{f"R_{col}": col for col in stat_cols},
    **{f"B_{col}": opp_name(col) for col in opp_stat_cols}
})


# Blue fighter rows
b_rows = raw_fights[
    ["B_fighter", "R_fighter", "B_DOB"]
    + shared_cols
    + [f"B_{col}" for col in stat_cols]
    + [f"R_{col}" for col in opp_stat_cols]
].copy()

b_rows = b_rows.rename(columns={
    "B_fighter": "fighter_name",
    "R_fighter": "opponent",
    "B_DOB": "DOB",
    **{f"B_{col}": col for col in stat_cols},
    **{f"R_{col}": opp_name(col) for col in opp_stat_cols}
})


# Combine red and blue rows
fighter_fights = pd.concat([r_rows, b_rows], ignore_index=True)


# Create result column
fighter_fights["winner"] = fighter_fights["winner"].fillna("").astype(str).str.strip()

fighter_fights["result"] = np.where(
    fighter_fights["winner"].eq(""),
    "Draw",
    np.where(
        fighter_fights["fighter_name"].eq(fighter_fights["winner"]),
        "Win",
        "Loss"
    )
)


# Reorder columns
ordered_cols = [
    "fighter_name",
    "DOB",
    "opponent",
    "result",
    "winner",
    "win_by",
    "last_round",
    "last_round_time",
    "date",
    "format",

    "KD",
    "SIG_STR_landed",
    "SIG_STR_att",
    "SIG_STR_pct",
    "TOTAL_STR_landed",
    "TOTAL_STR_att",
    "TD_landed",
    "TD_att",
    "TD_pct",
    "SUB_ATT",
    "REV",
    "CTRL",
    "HEAD_landed",
    "HEAD_att",
    "BODY_landed",
    "BODY_att",
    "LEG_landed",
    "LEG_att",
    "DISTANCE_landed",
    "DISTANCE_att",
    "CLINCH_landed",
    "CLINCH_att",
    "GROUND_landed",
    "GROUND_att",

    "SIG_STR_opp_landed",
    "SIG_STR_opp_att",
    "TOTAL_STR_opp_landed",
    "TOTAL_STR_opp_att",
    "TD_opp_landed",
    "TD_opp_att",
    "HEAD_opp_landed",
    "HEAD_opp_att",
    "BODY_opp_landed",
    "BODY_opp_att",
    "LEG_opp_landed",
    "LEG_opp_att",
    "DISTANCE_opp_landed",
    "DISTANCE_opp_att",
    "CLINCH_opp_landed",
    "CLINCH_opp_att",
    "GROUND_opp_landed",
    "GROUND_opp_att"
]

fighter_fights = fighter_fights[ordered_cols]

fighter_fights["date"] = pd.to_datetime(
    fighter_fights["date"],
    format="%B %d, %Y",
    errors="coerce"
)

# Sort by fighter name
fighter_fights = fighter_fights.sort_values(
    by=["fighter_name","date"],
    ascending=True
).reset_index(drop=True)

print("Shape:", fighter_fights.shape)
print(fighter_fights.head())

Shape: (17410, 52)
    fighter_name           DOB         opponent result           winner  \
0  AJ Cunningham  Sep 07, 1994    Ludovit Klein   Loss    Ludovit Klein   
1  AJ Cunningham  Sep 07, 1994      SuYoung You   Loss      SuYoung You   
2      AJ Dobson  Jan 18, 1992    Jacob Malkoun   Loss    Jacob Malkoun   
3      AJ Dobson  Jan 18, 1992  Armen Petrosyan   Loss  Armen Petrosyan   
4      AJ Dobson  Jan 18, 1992    Tafon Nchukwi    Win        AJ Dobson   

                 win_by  last_round last_round_time       date         format  \
0                KO/TKO           1            4:36 2024-03-02  3 Rnd (5-5-5)   
1  Decision - Unanimous           3            5:00 2025-03-15  3 Rnd (5-5-5)   
2  Decision - Unanimous           3            5:00 2022-02-12  3 Rnd (5-5-5)   
3  Decision - Unanimous           3            5:00 2022-10-22  3 Rnd (5-5-5)   
4  Decision - Unanimous           3            5:00 2023-08-12  3 Rnd (5-5-5)   

   ...  BODY_opp_landed  BODY_opp_att  LEG_

In [9]:
for x in list(fighter_fights.columns):
    print(x)
    print(fighter_fights[x].unique())

fighter_name
<StringArray>
[     'AJ Cunningham',          'AJ Dobson',        'AJ Fletcher',
         'Aalon Cruz',        'Aaron Brink',     'Aaron Phillips',
         'Aaron Pico',        'Aaron Riley',         'Aaron Rosa',
      'Aaron Simpson',
 ...
      'Zelim Imadaev', 'Zhalgas Zhumagulov',       'Zhang Lipeng',
     'Zhang Mingyang',      'Zhang Tiequan',        'Zhang Weili',
        'Zhu Kangjie',   'Zubaira Tukhugov',   'Zviad Lazishvili',
 'Zygimantas Ramaska']
Length: 2683, dtype: str
DOB
<StringArray>
['Sep 07, 1994', 'Jan 18, 1992', 'Feb 18, 1997', 'Sep 20, 1989',
 'Nov 12, 1974', 'Aug 05, 1989', 'Sep 23, 1996', 'Dec 09, 1980',
 'May 28, 1983', 'Jul 20, 1974',
 ...
 'Mar 24, 1994', 'Mar 09, 1996', 'Aug 02, 1984', 'Dec 10, 1983',
 'Jul 29, 1992', 'Jan 25, 1995', 'Aug 16, 1998', 'Jul 25, 1978',
 'Aug 13, 1989', 'Oct 25, 1991']
Length: 2288, dtype: str
opponent
<StringArray>
[       'Ludovit Klein',          'SuYoung You',        'Jacob Malkoun',
      'Armen Petrosyan', 

In [10]:
# Clean fighter-level fight data for career aggregation

# 1. Fix casing from older code versions if needed
fighter_fights = fighter_fights.rename(columns={
    "Winner": "winner",
    "Format": "format"
})


# 2. Group win_by into broader categories
# KO/TKO and Doctor Stoppage -> KO_TKO
# Submission -> Submission
# Decision types -> Decision
# Everything else, including Could Not Continue -> Other

win_by_map = {
    "KO/TKO": "KO_TKO",
    "TKO - Doctor's Stoppage": "KO_TKO",

    "Submission": "Submission",

    "Decision - Unanimous": "Decision",
    "Decision - Split": "Decision",
    "Decision - Majority": "Decision"
}

fighter_fights["win_by"] = (
    fighter_fights["win_by"]
    .fillna("")
    .astype(str)
    .str.strip()
    .map(win_by_map)
    .fillna("Other")
)


# 3. Convert time strings from mm:ss to seconds

def time_to_seconds(x):
    if pd.isna(x):
        return 0

    x = str(x).strip()

    if x == "":
        return 0

    parts = x.split(":")

    if len(parts) == 2:
        try:
            minutes = int(parts[0])
            seconds = int(parts[1])
            return minutes * 60 + seconds
        except ValueError:
            return 0

    return 0


# Convert last_round_time to seconds
fighter_fights["last_round_time"] = fighter_fights["last_round_time"].apply(time_to_seconds)


# 4. Convert format into total scheduled fight time in seconds

def format_to_total_scheduled_time(x):
    if pd.isna(x):
        return 0

    x = str(x).strip()

    if x == "No Time Limit":
        return 30 * 60

    if "(" not in x or ")" not in x:
        return 0

    try:
        inside_parentheses = x.split("(")[1].split(")")[0]
        round_lengths = inside_parentheses.split("-")
        total_minutes = sum(int(length) for length in round_lengths)
        return total_minutes * 60
    except ValueError:
        return 0


fighter_fights["total_scheduled_time"] = fighter_fights["format"].apply(format_to_total_scheduled_time)

# Drop original format column after extracting scheduled time
fighter_fights = fighter_fights.drop(columns=["format"], errors="ignore")


# 5. Convert CTRL from mm:ss to seconds

fighter_fights["CTRL"] = fighter_fights["CTRL"].apply(time_to_seconds)


# 6. Drop percentage columns because they will be recalculated later

pct_cols_to_drop = [
    "SIG_STR_pct",
    "TD_pct"
]

fighter_fights = fighter_fights.drop(columns=pct_cols_to_drop, errors="ignore")


print("Shape:", fighter_fights.shape)
print(fighter_fights.head())

Shape: (17410, 50)
    fighter_name           DOB         opponent result           winner  \
0  AJ Cunningham  Sep 07, 1994    Ludovit Klein   Loss    Ludovit Klein   
1  AJ Cunningham  Sep 07, 1994      SuYoung You   Loss      SuYoung You   
2      AJ Dobson  Jan 18, 1992    Jacob Malkoun   Loss    Jacob Malkoun   
3      AJ Dobson  Jan 18, 1992  Armen Petrosyan   Loss  Armen Petrosyan   
4      AJ Dobson  Jan 18, 1992    Tafon Nchukwi    Win        AJ Dobson   

     win_by  last_round  last_round_time       date  KD  ...  BODY_opp_att  \
0    KO_TKO           1              276 2024-03-02   0  ...            23   
1  Decision           3              300 2025-03-15   0  ...             5   
2  Decision           3              300 2022-02-12   0  ...             9   
3  Decision           3              300 2022-10-22   0  ...            39   
4  Decision           3              300 2023-08-12   0  ...            15   

   LEG_opp_landed  LEG_opp_att  DISTANCE_opp_landed  DISTANCE

# Save Fighter Fights Data

In [11]:
# Create total fight time in seconds
# Assumption: completed rounds before the final round are 5 minutes each

fighter_fights["total_fight_time"] = (
    (fighter_fights["last_round"] - 1) * 5 * 60
    + fighter_fights["last_round_time"]
)


print(
    fighter_fights[
        ["fighter_name", "last_round", "last_round_time", "total_fight_time"]
    ].head()
)



    fighter_name  last_round  last_round_time  total_fight_time
0  AJ Cunningham           1              276               276
1  AJ Cunningham           3              300               900
2      AJ Dobson           3              300               900
3      AJ Dobson           3              300               900
4      AJ Dobson           3              300               900


# Create Figther Career Statistics

In [12]:
# Create fighter_career_stats.csv from clean fighter-level fight data

# Simple total fight time assumption:
# completed rounds before final round = 5 minutes each
fighter_fights["last_round"] = pd.to_numeric(fighter_fights["last_round"], errors="coerce").fillna(0)
fighter_fights["last_round_time"] = pd.to_numeric(fighter_fights["last_round_time"], errors="coerce").fillna(0)

fighter_fights["total_fight_time"] = (
    (fighter_fights["last_round"] - 1) * 5 * 60
    + fighter_fights["last_round_time"]
)

fighter_fights["total_fight_time"] = fighter_fights["total_fight_time"].clip(lower=0)


# Helper functions
def safe_divide(num, denom):
    return np.where(denom > 0, num / denom, 0)

def safe_divide_nan(num, denom):
    return np.where(denom > 0, num / denom, np.nan)


# Normalize result and win_by values
fighter_fights["result"] = fighter_fights["result"].astype(str).str.strip()
fighter_fights["win_by"] = fighter_fights["win_by"].astype(str).str.strip()

win_by_clean = {
    "Decision": "DEC",
    "KO_TKO": "KO_TKO",
    "Submission": "SUB"
}

fighter_fights["win_by_clean"] = fighter_fights["win_by"].map(win_by_clean).fillna("Other")


# Create method indicators
fighter_fights["win_DEC"] = (
    (fighter_fights["result"] == "Win") &
    (fighter_fights["win_by_clean"] == "DEC")
).astype(int)

fighter_fights["win_KO_TKO"] = (
    (fighter_fights["result"] == "Win") &
    (fighter_fights["win_by_clean"] == "KO_TKO")
).astype(int)

fighter_fights["win_SUB"] = (
    (fighter_fights["result"] == "Win") &
    (fighter_fights["win_by_clean"] == "SUB")
).astype(int)

fighter_fights["loss_DEC"] = (
    (fighter_fights["result"] == "Loss") &
    (fighter_fights["win_by_clean"] == "DEC")
).astype(int)

fighter_fights["loss_KO_TKO"] = (
    (fighter_fights["result"] == "Loss") &
    (fighter_fights["win_by_clean"] == "KO_TKO")
).astype(int)

fighter_fights["loss_SUB"] = (
    (fighter_fights["result"] == "Loss") &
    (fighter_fights["win_by_clean"] == "SUB")
).astype(int)

fighter_fights["is_win"] = (fighter_fights["result"] == "Win").astype(int)
fighter_fights["is_loss"] = (fighter_fights["result"] == "Loss").astype(int)


# Numeric columns
# Source naming convention:
# - landed columns use "_landed"
# - attempted columns use "_att"
# - final output rate columns use "_land_per_min" and "_att_per_min"

numeric_cols = [
    "total_fight_time",
    "total_scheduled_time",
    "CTRL",

    "KD",
    "SUB_ATT",
    "REV",

    "SIG_STR_landed",
    "SIG_STR_att",
    "TOTAL_STR_landed",
    "TOTAL_STR_att",
    "TD_landed",
    "TD_att",

    "HEAD_landed",
    "HEAD_att",
    "BODY_landed",
    "BODY_att",
    "LEG_landed",
    "LEG_att",
    "DISTANCE_landed",
    "DISTANCE_att",
    "CLINCH_landed",
    "CLINCH_att",
    "GROUND_landed",
    "GROUND_att",

    "SIG_STR_opp_landed",
    "SIG_STR_opp_att",
    "TOTAL_STR_opp_landed",
    "TOTAL_STR_opp_att",
    "TD_opp_landed",
    "TD_opp_att",

    "HEAD_opp_landed",
    "HEAD_opp_att",
    "BODY_opp_landed",
    "BODY_opp_att",
    "LEG_opp_landed",
    "LEG_opp_att",
    "DISTANCE_opp_landed",
    "DISTANCE_opp_att",
    "CLINCH_opp_landed",
    "CLINCH_opp_att",
    "GROUND_opp_landed",
    "GROUND_opp_att"
]

for col in numeric_cols:
    fighter_fights[col] = pd.to_numeric(fighter_fights[col], errors="coerce").fillna(0)


# Aggregate totals by fighter
agg_cols = {
    "total_fight_time": "sum",
    "total_scheduled_time": "sum",
    "CTRL": "sum",

    "is_win": "sum",
    "is_loss": "sum",
    "win_DEC": "sum",
    "win_KO_TKO": "sum",
    "win_SUB": "sum",
    "loss_DEC": "sum",
    "loss_KO_TKO": "sum",
    "loss_SUB": "sum",

    "KD": "sum",
    "SUB_ATT": "sum",
    "REV": "sum",

    "SIG_STR_landed": "sum",
    "SIG_STR_att": "sum",
    "TOTAL_STR_landed": "sum",
    "TOTAL_STR_att": "sum",
    "TD_landed": "sum",
    "TD_att": "sum",

    "HEAD_landed": "sum",
    "HEAD_att": "sum",
    "BODY_landed": "sum",
    "BODY_att": "sum",
    "LEG_landed": "sum",
    "LEG_att": "sum",
    "DISTANCE_landed": "sum",
    "DISTANCE_att": "sum",
    "CLINCH_landed": "sum",
    "CLINCH_att": "sum",
    "GROUND_landed": "sum",
    "GROUND_att": "sum",

    "SIG_STR_opp_landed": "sum",
    "SIG_STR_opp_att": "sum",
    "TOTAL_STR_opp_landed": "sum",
    "TOTAL_STR_opp_att": "sum",
    "TD_opp_landed": "sum",
    "TD_opp_att": "sum",

    "HEAD_opp_landed": "sum",
    "HEAD_opp_att": "sum",
    "BODY_opp_landed": "sum",
    "BODY_opp_att": "sum",
    "LEG_opp_landed": "sum",
    "LEG_opp_att": "sum",
    "DISTANCE_opp_landed": "sum",
    "DISTANCE_opp_att": "sum",
    "CLINCH_opp_landed": "sum",
    "CLINCH_opp_att": "sum",
    "GROUND_opp_landed": "sum",
    "GROUND_opp_att": "sum"
}

fighter_career = fighter_fights.groupby("fighter_name").agg(agg_cols).reset_index()

fighter_counts = fighter_fights.groupby("fighter_name").size()
fighter_career["total_fights"] = fighter_career["fighter_name"].map(fighter_counts)

fighter_career = fighter_career.rename(columns={"fighter_name": "fighter"})

total_minutes = fighter_career["total_fight_time"] / 60


# Fight quickness
fighter_career["fight_quickness"] = safe_divide(
    fighter_career["total_fight_time"],
    fighter_career["total_scheduled_time"]
)


# Win/loss method percentages
fighter_career["win_DEC_pct"] = safe_divide(
    fighter_career["win_DEC"],
    fighter_career["is_win"]
)

fighter_career["win_KO_TKO_pct"] = safe_divide(
    fighter_career["win_KO_TKO"],
    fighter_career["is_win"]
)

fighter_career["win_SUB_pct"] = safe_divide(
    fighter_career["win_SUB"],
    fighter_career["is_win"]
)

fighter_career["loss_DEC_pct"] = safe_divide(
    fighter_career["loss_DEC"],
    fighter_career["is_loss"]
)

fighter_career["loss_KO_TKO_pct"] = safe_divide(
    fighter_career["loss_KO_TKO"],
    fighter_career["is_loss"]
)

fighter_career["loss_SUB_pct"] = safe_divide(
    fighter_career["loss_SUB"],
    fighter_career["is_loss"]
)


# Control percentage
fighter_career["CTRL_pct"] = safe_divide(
    fighter_career["CTRL"],
    fighter_career["total_fight_time"]
)


# Offensive accuracy
fighter_career["SIG_STR_acc"] = safe_divide(
    fighter_career["SIG_STR_landed"],
    fighter_career["SIG_STR_att"]
)

fighter_career["TD_acc"] = safe_divide(
    fighter_career["TD_landed"],
    fighter_career["TD_att"]
)

fighter_career["TOTAL_STR_acc"] = safe_divide(
    fighter_career["TOTAL_STR_landed"],
    fighter_career["TOTAL_STR_att"]
)


# Core per-minute rates
fighter_career["KD_per_min"] = safe_divide(fighter_career["KD"], total_minutes)
fighter_career["SUB_ATT_per_min"] = safe_divide(fighter_career["SUB_ATT"], total_minutes)
fighter_career["REV_per_min"] = safe_divide(fighter_career["REV"], total_minutes)


# Offensive per-minute rates
fighter_career["SIG_STR_land_per_min"] = safe_divide(fighter_career["SIG_STR_landed"], total_minutes)
fighter_career["SIG_STR_att_per_min"] = safe_divide(fighter_career["SIG_STR_att"], total_minutes)

fighter_career["TOTAL_STR_land_per_min"] = safe_divide(fighter_career["TOTAL_STR_landed"], total_minutes)
fighter_career["TOTAL_STR_att_per_min"] = safe_divide(fighter_career["TOTAL_STR_att"], total_minutes)

fighter_career["TD_land_per_min"] = safe_divide(fighter_career["TD_landed"], total_minutes)
fighter_career["TD_att_per_min"] = safe_divide(fighter_career["TD_att"], total_minutes)

fighter_career["HEAD_land_per_min"] = safe_divide(fighter_career["HEAD_landed"], total_minutes)
fighter_career["HEAD_att_per_min"] = safe_divide(fighter_career["HEAD_att"], total_minutes)

fighter_career["BODY_land_per_min"] = safe_divide(fighter_career["BODY_landed"], total_minutes)
fighter_career["BODY_att_per_min"] = safe_divide(fighter_career["BODY_att"], total_minutes)

fighter_career["LEG_land_per_min"] = safe_divide(fighter_career["LEG_landed"], total_minutes)
fighter_career["LEG_att_per_min"] = safe_divide(fighter_career["LEG_att"], total_minutes)

fighter_career["DISTANCE_land_per_min"] = safe_divide(fighter_career["DISTANCE_landed"], total_minutes)
fighter_career["DISTANCE_att_per_min"] = safe_divide(fighter_career["DISTANCE_att"], total_minutes)

fighter_career["CLINCH_land_per_min"] = safe_divide(fighter_career["CLINCH_landed"], total_minutes)
fighter_career["CLINCH_att_per_min"] = safe_divide(fighter_career["CLINCH_att"], total_minutes)

fighter_career["GROUND_land_per_min"] = safe_divide(fighter_career["GROUND_landed"], total_minutes)
fighter_career["GROUND_att_per_min"] = safe_divide(fighter_career["GROUND_att"], total_minutes)


# Defensive absorbed and opponent attempt rates
fighter_career["SIG_STR_absorbed_per_min"] = safe_divide(fighter_career["SIG_STR_opp_landed"], total_minutes)
fighter_career["SIG_STR_opp_att_per_min"] = safe_divide(fighter_career["SIG_STR_opp_att"], total_minutes)

fighter_career["SIG_STR_def"] = 1 - safe_divide(
    fighter_career["SIG_STR_opp_landed"],
    fighter_career["SIG_STR_opp_att"]
)

fighter_career["TOTAL_STR_absorbed_per_min"] = safe_divide(fighter_career["TOTAL_STR_opp_landed"], total_minutes)
fighter_career["TOTAL_STR_opp_att_per_min"] = safe_divide(fighter_career["TOTAL_STR_opp_att"], total_minutes)

fighter_career["TOTAL_STR_def"] = 1 - safe_divide(
    fighter_career["TOTAL_STR_opp_landed"],
    fighter_career["TOTAL_STR_opp_att"]
)

fighter_career["TD_absorbed_per_min"] = safe_divide(fighter_career["TD_opp_landed"], total_minutes)
fighter_career["TD_opp_att_per_min"] = safe_divide(fighter_career["TD_opp_att"], total_minutes)

fighter_career["TD_def"] = 1 - safe_divide_nan(
    fighter_career["TD_opp_landed"],
    fighter_career["TD_opp_att"]
)

fighter_career["HEAD_absorbed_per_min"] = safe_divide(fighter_career["HEAD_opp_landed"], total_minutes)
fighter_career["HEAD_opp_att_per_min"] = safe_divide(fighter_career["HEAD_opp_att"], total_minutes)

fighter_career["BODY_absorbed_per_min"] = safe_divide(fighter_career["BODY_opp_landed"], total_minutes)
fighter_career["BODY_opp_att_per_min"] = safe_divide(fighter_career["BODY_opp_att"], total_minutes)

fighter_career["LEG_absorbed_per_min"] = safe_divide(fighter_career["LEG_opp_landed"], total_minutes)
fighter_career["LEG_opp_att_per_min"] = safe_divide(fighter_career["LEG_opp_att"], total_minutes)

fighter_career["DISTANCE_absorbed_per_min"] = safe_divide(fighter_career["DISTANCE_opp_landed"], total_minutes)
fighter_career["DISTANCE_opp_att_per_min"] = safe_divide(fighter_career["DISTANCE_opp_att"], total_minutes)

fighter_career["CLINCH_absorbed_per_min"] = safe_divide(fighter_career["CLINCH_opp_landed"], total_minutes)
fighter_career["CLINCH_opp_att_per_min"] = safe_divide(fighter_career["CLINCH_opp_att"], total_minutes)

fighter_career["GROUND_absorbed_per_min"] = safe_divide(fighter_career["GROUND_opp_landed"], total_minutes)
fighter_career["GROUND_opp_att_per_min"] = safe_divide(fighter_career["GROUND_opp_att"], total_minutes)


# Final column order
final_cols = [
    "fighter",
    "total_fights",
    "total_fight_time",
    "fight_quickness",

    "win_DEC_pct",
    "win_KO_TKO_pct",
    "win_SUB_pct",
    "loss_DEC_pct",
    "loss_KO_TKO_pct",
    "loss_SUB_pct",

    "TOTAL_STR_acc",
    "SIG_STR_acc",
    "TD_acc",
    "CTRL_pct",

    "KD_per_min",
    "SUB_ATT_per_min",
    "REV_per_min",

    "SIG_STR_land_per_min",
    "SIG_STR_att_per_min",
    "SIG_STR_absorbed_per_min",
    "SIG_STR_opp_att_per_min",
    "SIG_STR_def",

    "TOTAL_STR_land_per_min",
    "TOTAL_STR_att_per_min",
    "TOTAL_STR_absorbed_per_min",
    "TOTAL_STR_opp_att_per_min",
    "TOTAL_STR_def",

    "TD_land_per_min",
    "TD_att_per_min",
    "TD_absorbed_per_min",
    "TD_opp_att_per_min",
    "TD_def",

    "HEAD_land_per_min",
    "HEAD_att_per_min",
    "HEAD_absorbed_per_min",
    "HEAD_opp_att_per_min",

    "BODY_land_per_min",
    "BODY_att_per_min",
    "BODY_absorbed_per_min",
    "BODY_opp_att_per_min",

    "LEG_land_per_min",
    "LEG_att_per_min",
    "LEG_absorbed_per_min",
    "LEG_opp_att_per_min",

    "DISTANCE_land_per_min",
    "DISTANCE_att_per_min",
    "DISTANCE_absorbed_per_min",
    "DISTANCE_opp_att_per_min",

    "CLINCH_land_per_min",
    "CLINCH_att_per_min",
    "CLINCH_absorbed_per_min",
    "CLINCH_opp_att_per_min",

    "GROUND_land_per_min",
    "GROUND_att_per_min",
    "GROUND_absorbed_per_min",
    "GROUND_opp_att_per_min"
]

fighter_career_stats = fighter_career[final_cols]

fighter_career_stats.to_csv("fighter_career_stats.csv", index=False)

print("Saved: fighter_career_stats.csv")
print("Shape:", fighter_career_stats.shape)
print(fighter_career_stats.head())

Saved: fighter_career_stats.csv
Shape: (2683, 56)
         fighter  total_fights  total_fight_time  fight_quickness  \
0  AJ Cunningham             2              1176         0.653333   
1      AJ Dobson             4              2973         0.825833   
2    AJ Fletcher             4              2769         0.769167   
3     Aalon Cruz             2               185         0.102778   
4    Aaron Brink             1                55         0.061111   

   win_DEC_pct  win_KO_TKO_pct  win_SUB_pct  loss_DEC_pct  loss_KO_TKO_pct  \
0          0.0             0.0          0.0      0.500000         0.500000   
1          1.0             0.0          0.0      0.666667         0.333333   
2          0.0             0.0          1.0      0.666667         0.000000   
3          0.0             0.0          0.0      0.000000         1.000000   
4          0.0             0.0          0.0      0.000000         0.000000   

   loss_SUB_pct  ...  DISTANCE_absorbed_per_min  DISTANCE_opp_att_

C:\Users\Abtin\AppData\Local\Temp\ipykernel_12848\739219972.py:337: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fighter_career["CLINCH_absorbed_per_min"] = safe_divide(fighter_career["CLINCH_opp_landed"], total_minutes)
C:\Users\Abtin\AppData\Local\Temp\ipykernel_12848\739219972.py:338: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fighter_career["CLINCH_opp_att_per_min"] = safe_divide(fighter_career["CLINCH_opp_att"], total_minutes)
C:\Users\Abtin\AppData\Local\Temp\ipykernel_12848\739219972.py:340: PerformanceWarning: DataFr